## Google Colab Setup

**GPU Required:** Before running, enable GPU runtime:
1. Go to **Runtime → Change runtime type**
2. Select **T4 GPU** (or better)
3. Click **Save**

In [1]:
# Install dependencies (skip if already done)
import os

# Set environment variables
os.environ['CCD_MIRROR_PATH'] = ''
os.environ['PDB_MIRROR_PATH'] = ''

if not os.path.isfile("FOUNDRY_READY"):
    print("Installing rc-foundry...")

    # Uninstall torchvision first to avoid operator conflicts
    os.system("pip uninstall -y torchvision")

    # Install rc-foundry
    os.system("pip install -q 'rc-foundry[all]'")

    # Mark as ready
    os.system("touch FOUNDRY_READY")

    print("Done!")
else:
    print("rc-foundry already installed.")

Installing rc-foundry...
Done!


In [2]:
# Download model weights (skips already-downloaded models automatically)
# In total, ~6GB (3GB for RFD3, 3GB for RF3, <100MB for MPNN); may take a few minutes depending on your connection speed
os.system("foundry install rfd3 ligandmpnn rf3")

0

# Example: End-To-End *De Novo* Protein Design Pipeline

## Overview

This notebook demonstrates an end-to-end protein design workflow using three deep learning networks from the Institute for Protein Design:

| Step | Model | Purpose |
|------|-------|---------|
| 1. **Generation** | RFD3 | Generate novel proteins via diffusion |
| 2. **Sequence Design** | MPNN | Design amino acid sequences for the generated backbone |
| 3. **Structure Validation via Refolding** | RF3 | Predict the structure from designed sequence to validate designability |

All models are unified through [AtomWorks](https://github.com/RosettaCommons/atomworks) (for both inference and training), relying on Biotite `AtomArray` objects.

### Pipeline Flow
```
RFD3 (backbone) → MPNN (sequence) → RF3 (validation) → RMSD comparison
```

---

In [3]:
import warnings
warnings.filterwarnings('ignore', module='atomworks')

# Shared utilities for visualization (from AtomWorks)
from atomworks.io.utils.visualize import view

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "user": "Session.username",


## Section 1: All-Atom Generation with RFD3

RFdiffusion3 (RFD3) generates *de novo* all-atom proteins that meet specific conditioning requirements.

**Parameters Used** *(many more are available for more complex protein design tasks)*:
- `length`: Target protein length in residues
- `diffusion_batch_size`: Number of structures to generate per batch
- `n_batches`: Number of batches to run

**Outputs:** Dictionary of `RFD3Output` objects.

In [23]:
from lightning.fabric import seed_everything
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine

# Set seed for reproducibility
seed_everything(0)

# Configure RFD3 inference
config = RFD3InferenceConfig(
    specification={
        'input': "/content/fold_basic_invasin_model_0.cif",
        'contig': "30-50,C113-120,30-80,/0,A1-377,/0,B1-480",
        'select_hotspots': "A70,A256,B160",
        'infer_ori_strategy': 'hotspots',
    },
    diffusion_batch_size=2,  # Generate 2 structures per batch
)

# Initialize engine and run generation
model = RFD3InferenceEngine(**config)
outputs = model.run(
    inputs=None,      # None for unconditional generation
    out_dir=None,     # None to return in memory (no file output)
    n_batches=2,      # Generate 1 batch
)

INFO: Seed set to 0
INFO:lightning.fabric.utilities.seed:Seed set to 0
INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO:rfd3.engine:[rank: 0] Finished inference batch in 62.14 seconds.
INFO:rfd3.engine:[rank: 0] Finished inference batch in 63.52 seconds.


In [24]:
# Inspect RFD3 outputs and extract the generated structures
#for idx, data in outputs.items():
#    print(f"Batch {idx}: {len(data)} structure(s)")
#    print(f"  Output type: {type(data[0]).__name__}")
#    print(f"  AtomArray: {data[0].atom_array}")

# Extract the first generated structure for downstream use
first_key = next(iter(outputs.keys()))
atom_array = outputs[first_key][0].atom_array

# Visualize the generated structure
view(atom_array)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

---

## Section 2: Sequence Design with MPNN

Protein and Ligand MPNN (Message Passing Neural Network) designs amino acid sequences that will fold into a target backbone structure.

**Model Options:**
- `protein_mpnn`: Original ProteinMPNN for protein-only design
- `ligand_mpnn`: Extended model supporting ligand-aware design

**Key Parameters:**
- `batch_size`: Number of sequences to generate per structure
- `remove_waters`: Whether to exclude water molecules from context

In [30]:
from mpnn.inference_engines.mpnn import MPNNInferenceEngine

# Configure MPNN inference engine
# See mpnn.utils.inference.MPNN_GLOBAL_INFERENCE_DEFAULTS for all options
engine_config = {
    "model_type": "ligand_mpnn",  # or "protein_mpnn" for vanilla ProteinMPNN
    "is_legacy_weights": True,    # Required for now for ligand_mpnn and protein_mpnn
    "out_directory": None,        # Return results in memory
    "write_structures": False,
    "write_fasta": False,
}

# Configure per-input inference options
# See mpnn.utils.inference.MPNN_PER_INPUT_INFERENCE_DEFAULTS for all options
input_configs = [
    {
        "batch_size": 8,         # Generate 10 sequences per structure
        "remove_waters": True,
    }
]

# Run sequence design on the RFD3-generated backbone
model = MPNNInferenceEngine(**engine_config)
mpnn_outputs = []
for first_key in outputs.keys():
  atom_array = outputs[first_key][0].atom_array
  mpnn_outputs.append([model.run(input_dicts=input_configs, atom_arrays=[atom_array]),int(first_key[1:])])

---

## Section 3: Structure Prediction with RF3

RF3 (RoseTTAFold 3) predicts protein structures from sequences. By re-folding the MPNN-designed sequence, we can validate whether the design is likely to adopt the intended backbone structure.

**Outputs:** `RF3Output` objects containing:
- `atom_array`: Predicted structure as Biotite AtomArray
- `summary_confidences`: Overall confidence metrics (pLDDT, PAE, pTM, etc.)
- `confidences`: Per-atom/residue confidence scores

**Confidence Metrics:**
| Metric | Description |
|--------|-------------|
| pLDDT | Per-residue confidence (0-1, higher is better) |
| PAE | Predicted Aligned Error (lower is better) |
| pTM | Predicted TM-score |
| ranking_score | Overall model quality score |

In [34]:
from rf3.inference_engines.rf3 import RF3InferenceEngine
from rf3.utils.inference import InferenceInput
from biotite.structure import get_residue_starts
from biotite.sequence import ProteinSequence

# Initialize RF3 inference engine
inference_engine = RF3InferenceEngine(ckpt_path='rf3', verbose=False)

# Create input from the MPNN-designed structure (first design)
# This re-folds the sequence to validate it adopts the intended structure
rf3_outputs = []
for j, mpnn_output_x in enumerate(mpnn_outputs):
  for i, item in enumerate(mpnn_output_x[0]):
    res_starts = get_residue_starts(item.atom_array)
    # Convert 3-letter codes to 1-letter using Biotite
    seq_1letter = ''.join(
        ProteinSequence.convert_letter_3to1(res_name)
        for res_name in item.atom_array.res_name[res_starts]
    )
    print(f"Sequence {mpnn_output_x[1]+1}_{i+1}: {seq_1letter}")
    input_structure = InferenceInput.from_atom_array(item.atom_array, example_id=f"{mpnn_output_x[1]+1}_{i+1}")
    rf3_outputs.append([inference_engine.run(inputs=input_structure),seq_1letter])

INFO:rf3.inference_engines.rf3:[rank: 0] Loading checkpoint from /root/.foundry/checkpoints/rf3_foundry_01_24_latest_remapped.ckpt...
INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO:rf3.inference_engines.rf3:[rank: 0] Found 1 structures to predict!
INFO:rf3.inference_engines.rf3:[rank: 0] Predicting structure 1/1: Sequence 1_1
INFO:rf3.inference_engines.rf3:[rank: 0] Found 1 structures to predict!
INFO:rf3.inference_engines.rf3:[rank: 0] Predicting structure 1/1: Sequence 1_2
INFO:rf3.inference_engines.rf3:[rank: 0] Found 1 structures to predict!
INFO:rf3.inference_engines.rf3:[rank: 0] Predicting structure 1/1: Sequence 1_3
INFO:rf3.inference_engines.rf3:[rank: 0] Found 1 structures to predict!
INFO:rf3.inference_engines.rf3:[rank: 0] Predicting structure 1/1: Sequence 1_4
INFO:rf3.inference_engines.rf3:[rank: 0] Found 1 structures to predict!
INFO:rf3.inference_engines.rf3:[rank: 0] Pr

In [43]:
import pandas as pd

# Prepare a list to collect data for the DataFrame
df_rows = []

# Loop through each RF3 output and collect the desired information
for sublist in rf3_outputs:
  rf3_output = sublist[0]
  row_data = rf3_output[list(rf3_output.keys())[0]][0].summary_confidences.copy() # Start with confidence metrics
  row_data['atom_array'] = rf3_output[list(rf3_output.keys())[0]][0].atom_array # Add the atom_array object
  row_data['name'] = list(rf3_output.keys())[0]
  # Assuming mpnn_outputs is structured as [[seq1, seq2, ...]]
  # and rf3_outputs elements correspond to mpnn_outputs[0][k]
  i,j = list(rf3_output.keys())[0].split(" ")[1].split("_")
  row_data['rfd3_num'] = int(i)
  row_data['mpnn_num'] = int(j)
  row_data['sequence'] = sublist[1]
  row_data['example_id'] = list(rf3_output.keys())[0] # Add the example_id for easy reference

  df_rows.append(row_data)

# Create the DataFrame from the collected rows
rf3_df = pd.DataFrame(df_rows)

# Display the first few rows of the DataFrame and its columns for verification
print("RF3 DataFrame created successfully!")
print(rf3_df.head())
print("\nDataFrame columns:", rf3_df.columns.tolist())

TypeError: 'int' object is not subscriptable

In [ ]:
# Extract the top-ranked prediction
rf3_output = rf3_df.loc[rf3_df['overall_plddt'].idxmax()]

# Visualize the predicted structure
view(rf3_output["atom_array"])

---

## Section 4: Validation and Export

The final step compares the RF3-predicted structure against the original RFD3-generated backbone. A low backbone RMSD indicates the designed sequence is likely to fold into the intended structure (high designability).

In [ ]:
from biotite.structure import rmsd, superimpose
from atomworks.constants import PROTEIN_BACKBONE_ATOM_NAMES
import numpy as np
from atomworks.io.utils.io_utils import to_cif_file

for line in rf3_df:
  # Get structures for comparison
  aa_generated = outputs["_"+line['rfd3_num']][0].atom_array              # Original RFD3 backbone (from Section 1)
  aa_refolded = line["atom_array"]    # RF3-predicted structure

  # Filter to backbone atoms (N, CA, C, O)
  bb_generated = aa_generated[np.isin(aa_generated.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)]
  bb_refolded = aa_refolded[np.isin(aa_refolded.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)]

  # Superimpose structures and calculate RMSD
  bb_refolded_fitted, _ = superimpose(bb_generated, bb_refolded)
  rmsd_value = rmsd(bb_generated, bb_refolded_fitted)
  rf3_df.loc["rmsd"] = rmsd_value

  print(f"Backbone RMSD: {rmsd_value:.2f} A")
  print(f"\nInterpretation: {'Excellent' if rmsd_value < 1.0 else 'Good' if rmsd_value < 2.0 else 'Moderate'} designability")
  to_cif_file(aa_refolded,line['example_id']+"_refolded.cif")